# Lesson 6 — APIs and HTTP

**The bridge from Assignment 1.** That assignment handed you dated *snapshots* —
a FRED CSV of Treasury yields, an Excel workbook of macro series — and asked you
to trust them. But those snapshots came from **somewhere**. Today you go to the
source: a program on your laptop asks a program on the internet for data, and gets
it back. By the end you can fetch those same series yourself.

An **API** (Application Programming Interface) is just a doorway a server opens so
that *other programs* — not humans clicking a website — can ask it for data in a
predictable shape. **HTTP** is the language they speak through that doorway.

```
Your code (the client)
        │  request:  "GET me series DGS10, as JSON"      (Unit 1)
        ▼
FRED's server
        │  response: status 200 + a JSON body            (Unit 1)
        ▼
requests turns bytes into a Python dict                  (Unit 2)
        ▼
JSON → DataFrame, types restored                         (Unit 2)
        ▼
handle rate limits, pages, timeouts; cache to disk       (Unit 3)
        ▼
read any API's docs; fetch a second series               (Unit 4)
```

**Prerequisites and setup**

- Lesson 2: loading data and *restoring the properties a file lost* — the exact
  move we make on a JSON response.
- Lesson 3 (bash/venv/git): why a secret like an API key lives in an environment
  variable, never in the code.
- Lesson 4 (Big-O): pagination is a linear scan over pages; a retry loop has a cost.
- Uses `requests` (already in the course environment) and `pandas`.

**This whole lesson runs offline.** Every `requests.get` below is a *real* HTTP
call with real status codes — but it goes to a tiny FRED stand-in running on your
own laptop, so you need no internet and no API key. Where a real key *would* change
the behaviour, the code is written so that setting `FRED_API_KEY` switches it to
the live FRED with no other change.

- Run top to bottom — later cells depend on earlier ones.

**Suggested sessions:** ① Units 1–2 · ② Units 3–4 + wrap-up

**Practice** lives in `../lesson6_exercise/` — you build a caching FRED client and a
pytest checker grades it, entirely offline.

In [ ]:
import json
import os
import time
import threading
import http.server
from pathlib import Path
from urllib.parse import urlparse, parse_qs

import pandas as pd
import requests

print("pandas", pd.__version__, "| requests", requests.__version__)

# The notebook writes only under ../data/ (gitignored); it creates nothing else.
Path("../data/api_cache").mkdir(parents=True, exist_ok=True)

# Fixtures pinned in the exercise folder, reshaped from Assignment 1's own data
# into the exact JSON shape FRED sends over the wire (see the fixtures manifest).
FIXTURES = Path("../lesson6_exercise/fixtures")
SERIES_IDS = ["DGS10", "CPIAUCSL", "UNRATE"]
print("series fixtures:", [f"{sid}.json" for sid in SERIES_IDS])

### Your offline FRED

Before Unit 1, we start a **stand-in for FRED on `127.0.0.1`** (your own machine).
It answers the same routes the real FRED does, so every request in this lesson is
a genuine HTTP round trip with genuine status codes — it simply never leaves your
laptop. Three routes matter:

- `/fred/series/observations?series_id=…` → the pinned series (**200**), or **404**
  for an unknown id;
- `/flaky` → **429** twice, then **200** (to practise polite retries in Unit 3);
- `/slow` → sleeps, so we can trigger a **timeout** in Unit 3.

You do not need to read every line — just know it is a real server, daemonised so
the notebook still exits cleanly.

In [ ]:
_FIXTURE = {sid: json.loads((FIXTURES / f"{sid}.json").read_text()) for sid in SERIES_IDS}
_server_state = {"observations_requests": 0, "flaky_hits": 0}

class FREDReplay(http.server.BaseHTTPRequestHandler):
    def log_message(self, *args):
        pass  # keep the notebook output clean

    def _json(self, code, payload, headers=None):
        body = json.dumps(payload).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json")
        for key, value in (headers or {}).items():
            self.send_header(key, value)
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        url = urlparse(self.path)
        query = parse_qs(url.query)
        if url.path == "/fred/series/observations":
            series_id = query.get("series_id", [""])[0]
            if series_id not in _FIXTURE:
                self._json(404, {"error_code": 404,
                                 "error_message": f"series_id {series_id!r} does not exist"})
                return
            _server_state["observations_requests"] += 1
            payload = _FIXTURE[series_id]
            offset = int(query.get("offset", [0])[0])
            limit = int(query.get("limit", [payload["count"]])[0])
            page = dict(payload)
            page["observations"] = payload["observations"][offset:offset + limit]
            page["offset"], page["limit"] = offset, limit
            self._json(200, page)
        elif url.path == "/flaky":
            _server_state["flaky_hits"] += 1
            if _server_state["flaky_hits"] <= 2:
                self._json(429, {"error_message": "Too Many Requests"}, headers={"Retry-After": "0"})
            else:
                self._json(200, {"ok": True, "served_on_attempt": _server_state["flaky_hits"]})
        elif url.path == "/slow":
            time.sleep(0.5)
            self._json(200, {"ok": True})
        else:
            self._json(404, {"error_code": 404, "error_message": "no such endpoint"})

_httpd = http.server.ThreadingHTTPServer(("127.0.0.1", 0), FREDReplay)
_httpd.daemon_threads = True
threading.Thread(target=_httpd.serve_forever, daemon=True).start()
LOCAL_FRED = f"http://127.0.0.1:{_httpd.server_address[1]}"
print("offline FRED stand-in serving at", LOCAL_FRED)

## Unit 1 — Client, server, request, response  ·  ~40 min

Two programs, two roles. The **client** starts the conversation by sending a
**request**; the **server** listens, does the work, and sends back a **response**.
Your browser is a client; so is `requests.get`. Nothing more mystical than a
question and an answer, both written as text in a shape both sides agreed on in
advance.

That agreed shape is the whole point of **HTTP**: a *protocol*, a contract that
says exactly what a request looks like, what a response looks like, and what the
status numbers mean. It is Lesson 1's idea at the level of two machines — **the
contract is a set of properties both sides promise to honour**, so neither has to
guess.

### Anatomy of a URL

A **URL** is the address of the thing you are asking for. Every piece answers one
question:

```
http://127.0.0.1:8000 /fred/series/observations ? series_id=DGS10 & file_type=json
└─scheme─┘└───host────┘└────────path───────────┘  └──────────query string────────┘
 protocol   which machine   which resource on it     parameters that refine the ask
```

The **path** names a resource (here, "observations"); the **query string** after
`?` carries parameters (`key=value`, joined by `&`) that narrow the request. Let
Python take one apart:

In [ ]:
example = f"{LOCAL_FRED}/fred/series/observations?series_id=DGS10&file_type=json"
parts = urlparse(example)

print("scheme :", parts.scheme)      # http vs https -> which protocol / is it encrypted
print("host   :", parts.netloc)      # which machine (here, your own laptop)
print("path   :", parts.path)        # which resource on that machine
print("query  :", parts.query)       # the parameters, still as one raw string
print()
print("query parsed into a dict:", parse_qs(parts.query))

### GET vs POST, headers, and status codes

Two verbs cover almost everything you will do:

| Verb | Means | Parameters go in | Think |
| ---- | ----- | ---------------- | ----- |
| `GET`  | *read* — fetch data, change nothing | the URL's query string | "show me DGS10" |
| `POST` | *send* — submit data to be stored/processed | a request **body** | "here is a new record" |

Fetching FRED data is always `GET`. Alongside the verb travel **headers** —
metadata *about* the message rather than the data itself: `Content-Type` (what
format the body is), `User-Agent` (who is asking), `Retry-After` (how long to wait
after a rate limit). And every response opens with a **status code**, a shared
three-digit vocabulary. The four you will actually meet:

| Code | Name | Meaning | Your move |
| ---- | ---- | ------- | --------- |
| 200 | OK | it worked, body has your data | use it |
| 404 | Not Found | that resource does not exist | fix the URL / id |
| 429 | Too Many Requests | you hit a rate limit | wait, then retry (Unit 3) |
| 500 | Server Error | the server broke, not you | retry later, then report |

The families generalise: **2xx** worked, **4xx** you asked wrong, **5xx** the
server failed.

In [ ]:
# Inspect a REAL request against the stand-in. This is a genuine HTTP round trip.
response = requests.get(
    f"{LOCAL_FRED}/fred/series/observations",
    params={"series_id": "DGS10", "file_type": "json"},
    timeout=10,
)
print("status code :", response.status_code, response.reason)     # 200 OK
print("content-type:", response.headers.get("Content-Type"))
print("body is JSON:", isinstance(response.json(), dict))
print("top-level keys:", list(response.json())[:6], "...")
print("observations the server reports:", response.json()["count"])

In [ ]:
# Ask for a series that does not exist -> the server answers 404, politely.
missing = requests.get(
    f"{LOCAL_FRED}/fred/series/observations",
    params={"series_id": "NOT_A_SERIES"},
    timeout=10,
)
print("status code:", missing.status_code, missing.reason)        # 404 Not Found
print("server's explanation:", missing.json()["error_message"])
print()
print("A 404 is not a crash -- it is a well-formed answer that says 'no'.")
print("The status code is how the client KNOWS, without parsing prose.")

🧭 Notice what made all of that work: neither side negotiated anything at run
time. The client already knew a response would carry a numeric status and (for a
200) a JSON body with a `count` and an `observations` list; the server already
knew a request would name a path and query parameters. **HTTP is that shared
contract** — the same "agree on the properties in advance so nobody guesses" idea
from Lesson 1, now spanning two computers that have never met.

## Unit 2 — requests and FRED  ·  ~55 min

Now we fetch a real economic series and turn it into a DataFrame. First, the
secret.

### API keys live in the environment, not in the code

FRED (like most APIs) wants an **API key** — a per-user password identifying who is
asking, so it can enforce rate limits. A key is a secret, and Lesson 3 told you
where secrets go: **into an environment variable, read at run time**, never typed
into a notebook and never committed to git.

```bash
export FRED_API_KEY=your_key_here      # once, in your shell, outside the code
```

The code reads it with `os.environ.get`. If it is absent — as in this offline
lesson — we fall back to the stand-in, and *the request code does not change*.

In [ ]:
FRED_API_KEY = os.environ.get("FRED_API_KEY")

if FRED_API_KEY:
    BASE_URL = "https://api.stlouisfed.org"     # the real FRED
    API_KEY = FRED_API_KEY
    print("FRED_API_KEY found -> requests will go to the LIVE FRED API.")
else:
    BASE_URL = LOCAL_FRED                         # our offline stand-in
    API_KEY = "OFFLINE_DEMO_KEY"
    print("No FRED_API_KEY set -> using the offline stand-in at", BASE_URL)

# The key line: EVERY request below uses BASE_URL and API_KEY. Only those two
# values differ between offline and live -- the logic is identical.

### Reading the docs to build the request

FRED's documentation for the `series/observations` endpoint lists its query
parameters. The few that matter today:

| Parameter | Required | What it does |
| --------- | -------- | ------------ |
| `series_id` | yes | which series, e.g. `DGS10` = 10-year Treasury yield |
| `api_key`   | yes | your key |
| `file_type` | no  | `json` to get JSON back (the default is XML) |
| `observation_start` | no | earliest date, `YYYY-MM-DD` |

Reading docs *is* the skill — you translate a table like this into a `params`
dict.

In [ ]:
params = {"series_id": "DGS10", "api_key": API_KEY, "file_type": "json"}

response = requests.get(f"{BASE_URL}/fred/series/observations", params=params, timeout=10)
response.raise_for_status()          # turn any 4xx/5xx into a loud error instead of silent bad data
payload = response.json()            # bytes -> Python dict

print("status:", response.status_code)
print("payload keys:", list(payload)[:6], "...")
print("count:", payload["count"])
print("first observation:", payload["observations"][0])

🧭 Look at that first observation: `{'date': '2021-01-04', 'value': '0.93', ...}`.
The value is a **string**, and missing days arrive as the string `"."`. JSON, like
a CSV, carries no types — it is Lesson 2's situation exactly. So loading a response
is the same two-step **restore the lost properties**: parse the dates, coerce the
numbers (letting `"."` become a visible `NaN`).

In [ ]:
# As it arrives: everything is text.
raw = pd.DataFrame(payload["observations"])[["date", "value"]]
print("straight off the wire (all text):", raw.dtypes.to_dict())
print(raw.head(3).to_dict("records"))
print()

# Restore the properties JSON dropped -- exactly Lesson 2's job on a loaded file.
dgs10 = raw.copy()
dgs10["date"] = pd.to_datetime(dgs10["date"])
dgs10["value"] = pd.to_numeric(dgs10["value"], errors="coerce")   # "." -> NaN, visibly

print("after restoring types:", dgs10.dtypes.to_dict())
print("rows:", len(dgs10), "| missing values flagged:", int(dgs10["value"].isna().sum()))

### Full circle: this is Assignment 1's snapshot

Assignment 1 shipped `DGS10.csv` as a pinned snapshot and never said where it came
from. It came from **this endpoint**. The offline fixture we just fetched was
reshaped, value-for-value, from that very CSV — so the fetched series and the
Assignment 1 snapshot must be identical.

In [ ]:
snapshot = pd.read_csv("../pandas-finance-assignment/data_offline/DGS10.csv")
snapshot_2021 = snapshot[snapshot["DATE"] >= "2021-01-01"].reset_index(drop=True)

same_dates = raw["date"].tolist() == snapshot_2021["DATE"].tolist()
same_values = raw["value"].tolist() == snapshot_2021["DGS10"].astype(str).tolist()

print("Assignment 1 snapshot rows:", len(snapshot_2021))
print("freshly fetched rows      :", len(raw))
print("fetched series == Assignment 1 snapshot, exactly:", same_dates and same_values)
print()
print("The snapshot was never magic. It was one GET request, saved to a file.")

In [ ]:
import matplotlib.pyplot as plt

monthly_yield = (dgs10.dropna(subset=["value"])
                       .set_index("date")["value"].resample("MS").mean())

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(monthly_yield.index, monthly_yield.values, linewidth=2)
ax.set(title="10-year Treasury yield, monthly average (fetched offline)",
       xlabel="month", ylabel="percent")
ax.grid(True, alpha=.3)
plt.show()

---
## ⏱️ Session 2 warm-up  ·  5 questions from last time

Answer from memory first, then check.

1. In a request/response exchange, which side is the client and which is the
   server — and which one starts the conversation?
2. What does each of 200, 404, and 429 tell you?
3. `GET` vs `POST`: which one reads data, and where do its parameters travel?
4. Why does an API key belong in an environment variable instead of in the notebook?
5. A FRED response arrives as JSON with every value a string and `"."` for missing.
   Which two Lesson-2 steps turn it into a trustworthy DataFrame?

<details><summary><b>Answers</b></summary>

1. The client (your code, `requests.get`) starts by sending a request; the server
   listens and answers. The client always begins.
2. 200 = it worked, use the body. 404 = that resource/id does not exist. 429 = you
   hit the rate limit; wait and retry.
3. `GET` reads, and its parameters ride in the URL's query string. (`POST` sends a
   body.)
4. A key is a secret; environment variables keep it out of the code and out of git,
   so it is never committed or shared by accident.
5. Parse the dates (`to_datetime`) and coerce the numbers (`to_numeric`,
   `errors="coerce"`) so `"."` becomes a visible `NaN` — restoring the types JSON
   could not carry.
</details>

## Unit 3 — The rough edges  ·  ~45 min

The happy path — one request, a 200, clean JSON — is the exception on real
networks. Four rough edges separate a toy script from a client you can rely on:
**rate limits, pagination, timeouts, and caching.**

### Rate limits and polite retries

A server protects itself by capping how often you may call it (FRED allows roughly
120 requests per minute). Cross the line and you get **429 Too Many Requests**. The
polite response is not to give up and not to hammer — it is to **wait and retry,
backing off a little longer each time** (exponential backoff). Our `/flaky`
endpoint returns 429 twice, then 200, so we can watch it work.

In [ ]:
def get_with_retries(url, max_attempts=5, base_delay=0.1):
    reply = None
    for attempt in range(max_attempts):
        reply = requests.get(url, timeout=10)
        if reply.status_code != 429:
            return reply                       # success (or a real error) -> stop
        wait = base_delay * (2 ** attempt)     # 0.1s, 0.2s, 0.4s, ... exponential backoff
        print(f"  attempt {attempt + 1}: got 429, backing off {wait:.2f}s then retrying")
        time.sleep(wait)
    return reply

result = get_with_retries(f"{LOCAL_FRED}/flaky")
print("final status:", result.status_code, "->", result.json())

### Pagination: a loop over pages

A server will not hand you a million rows in one response — it returns a **page** at
a time. You ask for a window with `offset` and `limit`, then loop, advancing the
offset until you have collected all `count` rows. It is Lesson 4's **linear scan**,
one page per step.

In [ ]:
def fetch_all_pages(series_id, page_size=500):
    collected, offset = [], 0
    while True:
        reply = requests.get(
            f"{LOCAL_FRED}/fred/series/observations",
            params={"series_id": series_id, "offset": offset, "limit": page_size},
            timeout=10,
        )
        body = reply.json()
        collected.extend(body["observations"])
        print(f"  page at offset {offset:>4}: {len(body['observations']):>3} rows "
              f"(total so far {len(collected)})")
        offset += page_size
        if offset >= body["count"]:
            break
    return collected

all_rows = fetch_all_pages("DGS10")
print("reassembled", len(all_rows), "rows from pages")

### Timeouts: never wait forever

If a server hangs, a request with no time limit freezes your whole program. **Always
pass `timeout=`.** When it is exceeded, `requests` raises — which you can catch —
instead of blocking. Our `/slow` route sleeps half a second; we give it only 0.2s.

In [ ]:
try:
    requests.get(f"{LOCAL_FRED}/slow", timeout=0.2)      # the endpoint sleeps 0.5s
except requests.exceptions.Timeout:
    print("Timeout raised after 0.2s -- caught it; the pipeline stays responsive.")

### Caching to disk — and which format?

Every later lesson reuses this move: **fetch once, save the response, and next time
read the saved copy** instead of hitting the network. That is what keeps this whole
course runnable offline.

Which format for the cache? Lesson 2 already answered it. Store the **raw JSON the
server sent**: JSON preserves the response's exact nested shape, re-parses to an
identical dict, and stays human-readable — so a cached hit is *indistinguishable*
from a live fetch. (Parquet would win for a big *tidied table*; but a raw API
response is a nested document, and JSON is what it already is.)

In [ ]:
CACHE = Path("../data/api_cache")

def fetch_with_cache(series_id):
    cache_path = CACHE / f"{series_id}.json"
    if cache_path.exists():
        return json.loads(cache_path.read_text()), "cache"          # served from disk
    reply = requests.get(
        f"{BASE_URL}/fred/series/observations",
        params={"series_id": series_id, "api_key": API_KEY, "file_type": "json"},
        timeout=10,
    )
    reply.raise_for_status()
    cache_path.write_text(reply.text)                                # store the EXACT bytes
    return reply.json(), "network"

# Start clean so the demo is reproducible on a re-run.
if (CACHE / "UNRATE.json").exists():
    (CACHE / "UNRATE.json").unlink()

before = _server_state["observations_requests"]
_, source_first = fetch_with_cache("UNRATE")
_, source_second = fetch_with_cache("UNRATE")
after = _server_state["observations_requests"]

print("first call served from :", source_first)      # network
print("second call served from:", source_second)     # cache
print("times the server was contacted for two calls:", after - before)
print("cache file now on disk:", (CACHE / "UNRATE.json").exists())

🧭 The **cache-first-with-fallback** pattern — try the cache, else fetch and save —
is the backbone of every offline-capable pipeline, and it is precisely what you
build in this lesson's exercise. Fetch the truth once; depend on the copy.

## Unit 4 — Reading API documentation  ·  ~30 min

The real skill is not memorising FRED — it is being handed *any* API's docs and
constructing the request yourself. Here is a documentation excerpt for a **second
series**, the kind you would find on FRED's site:

> **Series `UNRATE` — Unemployment Rate**
> Percent of the labour force without a job, **monthly**, seasonally adjusted.
> Retrieved from the same `fred/series/observations` endpoint; only `series_id`
> changes. Values are strings; there is one observation per month.

That is enough to build the request. Your guided task: fetch `UNRATE` (through the
cache), then tidy it to a monthly `month`/`value` frame the way you tidied `DGS10`.

In [ ]:
def tidy_monthly(payload):
    frame = pd.DataFrame(payload["observations"])[["date", "value"]]
    frame["date"] = pd.to_datetime(frame["date"])
    frame["value"] = pd.to_numeric(frame["value"], errors="coerce")   # "." -> NaN
    frame = frame.dropna(subset=["value"])
    monthly = (frame.set_index("date")["value"]
                    .resample("MS").mean()      # one row per month, at its first day
                    .dropna()
                    .reset_index())
    monthly.columns = ["month", "value"]
    return monthly

unrate_payload, served_from = fetch_with_cache("UNRATE")     # you already cached this above
unrate = tidy_monthly(unrate_payload)

print("UNRATE served from:", served_from)
print(unrate.head())
print("...")
print(unrate.tail(3).to_string(index=False))
print("monthly rows:", len(unrate))

### SDK vs raw HTTP — one honest paragraph

Many APIs also ship an **SDK** (a wrapper library — for FRED there is `fredapi`,
and `pandas-datareader`). An SDK can be lovely: `fred.get_series("UNRATE")` in one
line, no URL to build. But it is a convenience layer over the exact `GET` you just
wrote — and it can lag behind the API, hide which parameters exist, and add a
dependency you must trust and update. Raw `requests` is universal and transparent:
if you can read the docs, you can call *any* HTTP API, SDK or not. Same Lesson 4
lesson as `max()` over the hand-written loop — **the convenience runs the procedure
for you; it never removes it.** Learn the request first; reach for the SDK once you
know what it is doing on your behalf.

## Wrap-up

### What you can do now

- Read a URL as scheme · host · path · query, and build a request from a docs table.
- Send a real `GET` with `requests`, check the **status code**, and `raise_for_status()`.
- Turn a JSON response into a DataFrame and **restore the types** the wire dropped.
- Survive the rough edges: **retry** a 429 with backoff, **paginate** a large
  result, guard every call with a **timeout**, and **cache** responses to disk.
- Read an unfamiliar endpoint's docs and fetch a new series unaided.

### Where the through-lines landed

- *Lesson 1 — properties are a contract*: HTTP is that contract between two
  machines — agreed request shape, response shape, and status vocabulary.
- *Lesson 2 — storage is not neutral; restore lost properties*: a JSON response
  carries no types, so we parse dates and coerce numbers, exactly as with a file —
  and we cache as JSON because it is faithful to the response's shape.
- *Lesson 3 — secrets and convenience*: the API key lives in an environment
  variable; an SDK hides the request but never removes it.
- *Lesson 4 — cost has a shape*: pagination is a scan over pages; a retry loop and
  a timeout are how you bound that cost.

### The lesson, one sentence

> *An API is a doorway and HTTP is the contract; you send a `GET`, check the status,
> restore the types, and — because the network is unreliable — retry, paginate, time
> out, and cache, so the data you fetched once is yours to keep.*

### What comes next

Assignment 1's snapshots are no longer a mystery: you can regenerate them. Next in
the course, those tidy frames flow into **databases**, where Lesson 7 gives them
a schema, constraints, and durable SQLite storage.

### Practice

Open **`../lesson6_exercise/`**: implement `get_series` (cache → live → fixture)
and `tidy_monthly`; a pytest checker grades both offline.
Run it with `python3 -m pytest test_lesson6.py -v` from inside the folder.

### Extensions

- [FRED API — `series/observations`](https://fred.stlouisfed.org/docs/api/fred/series_observations.html) — the endpoint used all lesson
- [FRED API overview](https://fred.stlouisfed.org/docs/api/fred/) — every endpoint and parameter
- [`requests` documentation](https://requests.readthedocs.io/en/latest/) — the library you called
- [MDN — HTTP overview](https://developer.mozilla.org/en-US/docs/Web/HTTP) — methods, status codes, headers in depth

**Next:** the practice exercise in `../lesson6_exercise/` — a caching FRED client, graded offline.